# Nova AI — نموذج توليد الصور الخاص بنا (Stable Diffusion، مملوك بالكامل)

**منفصل تماماً عن دفتر `merge_and_finetune.ipynb`** (فهم النص والصور) —
فهم الصور وتوليدها بنيتان مختلفتان جذرياً في الشبكات العصبية، فلا يمكن
لأي قدر من تدريب Qwen2.5-VL أن يكسبه القدرة على توليد صور. هذا نموذج
**ثانٍ ومستقل بالكامل**، لكنه بنفس مبدأ الملكية بالضبط: أوزان مفتوحة
المصدر بالكامل، نُنزّلها، نملك نسختنا الخاصة على مستودعنا، ونستضيفها
مجاناً — وليس استدعاء API مستأجَراً من أي شركة.

**لا حاجة لتشغيل هذا الدفتر أسبوعياً** كدفتر النص/الرؤية — نموذج
Stable Diffusion المُدرَّب مسبقاً يعمل بجودة عالية فور تنزيله دون أي
تدريب إضافي، ولا يوجد مصدر بيانات "يكبر تلقائياً" لتوليد الصور كما
يحدث مع محادثات نوفا النصية. شغّله **مرة واحدة** لتفعيل الميزة، ولاحقاً
مرة أو مرتين فقط إن أردت لاحقاً تخصيص أسلوب بصري مميز لنوفا عبر تدريب
LoRA خفيف (خطوة مستقبلية اختيارية، غير مبنية في هذا الدفتر بعد).

## الإعداد لمرة واحدة فقط

**1) استورد الدفتر** بنفس طريقة الدفتر الآخر تماماً: Kaggle → Create →
New Notebook → File → Import Notebook → GitHub → الصق رابط هذا الملف.
فعّل **GPU T4** من Notebook options.

**2) الأسرار (Secrets):** فقط `HF_TOKEN` و`HF_USERNAME` (نفس القيمتين
المستخدمتين في دفتر النص/الرؤية — إن كان هذا الدفتر في نفس حساب
Kaggle، الأسرار مشتركة تلقائياً ولا حاجة لإضافتها مجدداً).

**3) شغّل الخلايا بالترتيب من الأعلى للأسفل يدوياً هذه المرة** — لا
حاجة لجدولة أسبوعية (Schedule) لهذا الدفتر تحديداً، فقط شغّله عند
الحاجة.

**بعد نجاح الرفع (آخر خلية):** ضع اسم المستودع الذي تطبعه في
`HF_IMAGE_MODEL_ID` على Render (Environment Variables) ثم Manual
Deploy. عندها يعمل أمر `/صورة` في بوت نوفا فوراً.

In [ ]:
# الخلية 1 — تثبيت الأدوات
!pip install -q diffusers accelerate transformers safetensors huggingface_hub

In [ ]:
# الخلية 2 — تسجيل الدخول إلى Hugging Face (بلا أي تدخل يدوي)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("تم تسجيل الدخول إلى Hugging Face بنجاح")

In [ ]:
# الخلية 3 — الأساس المفتوح، ومستودعنا الخاص
from kaggle_secrets import UserSecretsClient

HF_USERNAME = UserSecretsClient().get_secret("HF_USERNAME")
REPO_ID = f"{HF_USERNAME}/nova-image-gen"
print("سيُرفَع نموذجنا إلى:", REPO_ID)

In [ ]:
# الخلية 4 — تحميل النموذج (بمحاولة عدة أسماء بديلة) والتحقق من عمله فعلياً
#
# أسماء مستودعات Stable Diffusion على Hugging Face تتغيّر بمرور الوقت
# (بالضبط كما حدث مع نموذج Gemini سابقاً في دفتر النص) — بدل الاعتماد
# على اسم واحد قد يُنقَل أو يُعاد تسميته، نجرّب عدة مستودعات معروفة
# ومستقرة تاريخياً بالترتيب حتى ينجح أحدها. إن فشلت كلها، تحقق يدوياً
# من huggingface.co/models?search=stable-diffusion لاسم حالي وأضفه
# لهذه القائمة.
import torch
from diffusers import AutoPipelineForText2Image

_CANDIDATE_IMAGE_MODELS = [
    "stabilityai/stable-diffusion-2-1",
    "stabilityai/stable-diffusion-2-1-base",
    "runwayml/stable-diffusion-v1-5",
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "CompVis/stable-diffusion-v1-4",
    "stabilityai/sd-turbo",
]

pipe = None
BASE_IMAGE_MODEL_ID = None
for _candidate in _CANDIDATE_IMAGE_MODELS:
    try:
        print("تجربة:", _candidate, "...")
        pipe = AutoPipelineForText2Image.from_pretrained(_candidate, torch_dtype=torch.float16)
        BASE_IMAGE_MODEL_ID = _candidate
        print("نجح التحميل من:", _candidate)
        break
    except Exception as e:
        print("فشل", _candidate, "-", type(e).__name__)

if pipe is None:
    raise RuntimeError(
        "فشلت كل الأسماء المرشّحة. تحقق يدوياً من huggingface.co/models?search=stable-diffusion "
        "عن اسم مستودع حالي وأضفه إلى _CANDIDATE_IMAGE_MODELS أعلاه ثم أعد تشغيل هذه الخلية."
    )

pipe = pipe.to("cuda")

test_image = pipe("a friendly cartoon robot mascot, simple flat design", num_inference_steps=25).images[0]
test_image.save("test_output.png")
print("تم توليد صورة تجريبية بنجاح باستخدام", BASE_IMAGE_MODEL_ID, "— تحقق من test_output.png في ملفات الجلسة (Output) للتأكد بصرياً.")

In [ ]:
# الخلية 5 — رفع النموذج إلى مستودعنا الخاص على Hugging Face Hub
#
# نسخة كاملة من أوزان Stable Diffusion المفتوحة (BASE_IMAGE_MODEL_ID
# الذي نجح تحميله في الخلية السابقة)، منسوخة بالكامل إلى مستودعنا
# نحن — نملك نسختنا الآن بشكل كامل ومستقل عن أي تغيير مستقبلي على
# المستودع الأصلي، وجاهزة لاستضافة HF Inference المجانية بنفس طريقة
# نموذج النص/الرؤية بالضبط.
pipe.save_pretrained("./nova-image-gen-local")

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path="./nova-image-gen-local", repo_id=REPO_ID)
print(f"تم الرفع: https://huggingface.co/{REPO_ID}")
print("ضع هذا في HF_IMAGE_MODEL_ID داخل ai-system/.env أو Render:", REPO_ID)

---
## نموذج توليد الفيديو الخاص بنا (CogVideoX-2B، مملوك بالكامل)

**owner spec 2026-09-08:** نفس المبدأ بالضبط — لا نستأجر توليد فيديو من
أي شركة، بل ننزّل نموذجاً مفتوح الأوزان بالكامل ونملك نسختنا الخاصة.

**لماذا CogVideoX-2B تحديداً؟** بحثت هذا حياً (وليس تخميناً): إصداره
2B مرخّص Apache 2.0 (استخدام تجاري حر بالكامل) ومؤكَّد أنه يعمل على
بطاقات 16GB — بالضبط ما توفره Kaggle's T4 مجاناً. توليد الفيديو أثقل
بكثير من توليد الصور (كل فيديو هو عشرات الإطارات، لا إطار واحد)، لذا
الإصدار الأخف (2B) هو الخيار الواقعي الوحيد على عتاد مجاني، وليس
إصداره الأكبر (5B) الذي يحتاج ذاكرة أكبر مما نملك هنا.

**بديل احتياطي أخف حتى من ذلك:** `damo-vilab/text-to-video-ms-1.7b`
(نموذج ModelScope نفسه — نستضيف عليه أصلاً خادم النص/الرؤية الحي)،
أصغر وأضمن نجاحاً إن فشل تحميل CogVideoX-2B لأي سبب (مساحة، ذاكرة،
تغيّر اسم المستودع بمرور الوقت كما حدث سابقاً مع Stable Diffusion
أعلاه وGemini في الدفتر الآخر).

**صادق حول الجودة والسرعة:** فيديو 2B على معالج رسومي مجاني لن ينافس
منتجات تجارية مدفوعة (Sora، Veo) — هذا نموذج مفتوح صغير نسبياً، ناتجه
قصير (بضع ثوانٍ) ومتواضع الدقة مقارنة بها. الهدف هنا: ملكية حقيقية
وميزة تعمل فعلياً بتكلفة صفر، وليس منافسة أفضل الحلول التجارية.

**نفس أسلوب تشغيل الصور تماماً: مرة واحدة يدوياً، ليس أسبوعياً.**

In [ ]:
# الخلية 6 — تثبيت أدوات الفيديو الإضافية
#
# diffusers الحديثة تدعم CogVideoXPipeline (لم تكن مثبَّتة بعد لخلية
# الصور أعلاه بإصدار حديث بما يكفي دائماً) — نحدّثها صراحة هنا.
# imageio[ffmpeg] هو ما يحوّل الإطارات الناتجة إلى ملف mp4 فعلي عبر
# diffusers.utils.export_to_video.
!pip install -q -U diffusers transformers accelerate
!pip install -q "imageio[ffmpeg]"

In [ ]:
# الخلية 7 — الأساس المفتوح لتوليد الفيديو، ومستودعنا الخاص
from kaggle_secrets import UserSecretsClient

VIDEO_REPO_ID = f"{HF_USERNAME}/nova-video-gen"
print("سيُرفَع نموذج الفيديو إلى:", VIDEO_REPO_ID)

In [ ]:
# الخلية 8 — تحميل نموذج الفيديو (بمحاولة مرشَّحين مختلفي البنية) والتحقق الفعلي بتوليد فيديو حقيقي
#
# على عكس خلية الصور أعلاه (كل مرشحيها من عائلة Stable Diffusion،
# فتشترك في نفس فئة الأنبوب AutoPipelineForText2Image)، مرشَّحا الفيديو
# هنا بنيتان مختلفتان فعلياً (CogVideoX له فئة أنبوب خاصة به)، فكل
# مرشح يحمل فئته وطريقة استدعائه الخاصة بدل حلقة موحّدة واحدة.
#
# enable_model_cpu_offload/enable_vae_slicing/enable_vae_tiling: هذه
# ليست تحسينات اختيارية هنا بل ما يجعل نموذجاً بحجم CogVideoX-2B يعمل
# أصلاً ضمن 16GB بدل نفاد الذاكرة (OOM) — موثَّقة على صفحة النموذج
# الرسمية لتشغيله على عتاد بذاكرة محدودة.
import torch
from diffusers.utils import export_to_video

video_pipe = None
BASE_VIDEO_MODEL_ID = None
VIDEO_KIND = None  # "cogvideox" | "text2video_ms" — تحدد طريقة الاستدعاء لاحقاً


def _try_cogvideox():
    from diffusers import CogVideoXPipeline

    pipe = CogVideoXPipeline.from_pretrained("THUDM/CogVideoX-2b", torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()
    return pipe, "THUDM/CogVideoX-2b", "cogvideox"


def _try_text2video_ms():
    from diffusers import DiffusionPipeline

    pipe = DiffusionPipeline.from_pretrained(
        "damo-vilab/text-to-video-ms-1.7b", torch_dtype=torch.float16, variant="fp16"
    )
    pipe.enable_model_cpu_offload()
    return pipe, "damo-vilab/text-to-video-ms-1.7b", "text2video_ms"


for _attempt in (_try_cogvideox, _try_text2video_ms):
    try:
        print("تجربة:", _attempt.__name__, "...")
        video_pipe, BASE_VIDEO_MODEL_ID, VIDEO_KIND = _attempt()
        print("نجح التحميل من:", BASE_VIDEO_MODEL_ID)
        break
    except Exception as e:
        print("فشل", _attempt.__name__, "-", type(e).__name__, "-", str(e)[:200])

if video_pipe is None:
    raise RuntimeError(
        "فشل تحميل كلا مرشَّحَي الفيديو. تحقق يدوياً من huggingface.co/models?pipeline_tag=text-to-video "
        "عن اسم مستودع حالي، وأضف دالة محاولة جديدة أعلاه على نمط الدالتين الموجودتين."
    )

_test_prompt = "a friendly cartoon robot mascot waving hello, simple flat design"
if VIDEO_KIND == "cogvideox":
    _frames = video_pipe(
        prompt=_test_prompt, num_videos_per_prompt=1, num_inference_steps=50, num_frames=49, guidance_scale=6
    ).frames[0]
    export_to_video(_frames, "test_output_video.mp4", fps=8)
else:
    _frames = video_pipe(prompt=_test_prompt, num_inference_steps=25, num_frames=16).frames[0]
    export_to_video(_frames, "test_output_video.mp4")

print("تم توليد فيديو تجريبي بنجاح باستخدام", BASE_VIDEO_MODEL_ID, "— تحقق من test_output_video.mp4 في ملفات الجلسة (Output) للتأكد بصرياً.")

In [ ]:
# الخلية 9 — رفع نموذج الفيديو إلى مستودعنا الخاص على Hugging Face Hub
video_pipe.save_pretrained("./nova-video-gen-local")

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(VIDEO_REPO_ID, exist_ok=True)
api.upload_folder(folder_path="./nova-video-gen-local", repo_id=VIDEO_REPO_ID)
print(f"تم الرفع: https://huggingface.co/{VIDEO_REPO_ID}")
print("النوع (لضبط HF_VIDEO_MODEL_KIND على Render):", VIDEO_KIND)
print("ضع هذا في HF_VIDEO_MODEL_ID داخل ai-system/.env أو Render:", VIDEO_REPO_ID)